### Machine Learning - Capstone Project 1
### Singapore Housing Resale Price Prediction
### 02 - Pre Processing & Model Building

In [1]:
# Import required libraies
import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import r2_score

In [2]:
dataset = pd.read_csv("Dataset-Singapore-HDB.csv")

In [3]:
# shape property will give the (rows x columns) of the dataset
# This dataset has 232,188 rows and 15 columns
dataset.shape

(232188, 15)

In [4]:
# Give all columns
dataset.columns

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range',
       'floor_area_sqm', 'floor_area_sqft', 'flat_model',
       'lease_commence_date', 'house_age', 'remaining_lease', 'price_per_sqm',
       'price_per_sqft', 'resale_price'],
      dtype='str')

---

### Data pre-processing

In [5]:
# Drop less important columns
dataset.drop(columns=["month", "block", "street_name", "remaining_lease"], inplace=True)

In [6]:
# Since model cannot understand the categorical data, needs to encode before processing.
dataset = pd.get_dummies(dataset, dtype=int, drop_first=True)

In [7]:
dataset

,floor_area_sqm,floor_area_sqft,lease_commence_date,house_age,price_per_sqm,price_per_sqft,resale_price,town_BEDOK,town_BISHAN,town_BUKIT BATOK,...,flat_model_Multi Generation,flat_model_New Generation,flat_model_Premium Apartment,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
0,44.0,473.6116,1979,47,5272.727273,489.852867,232000.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,67.0,721.1813,1978,48,3731.343284,346.653470,250000.0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,67.0,721.1813,1980,46,3910.447761,363.292836,262000.0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,68.0,731.9452,1980,46,3897.058824,362.048962,265000.0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,67.0,721.1813,1980,46,3955.223881,367.452678,265000.0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232183,146.0,1571.5294,1988,38,6815.068493,633.141193,995000.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
232184,142.0,1528.4738,1987,39,6901.408451,641.162446,980000.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
232185,145.0,1560.7655,1987,39,6620.689655,615.082791,960000.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
232186,145.0,1560.7655,1988,38,7371.641379,684.848557,1068888.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
dataset.shape

(232188, 74)

---

### Prepare Training & Test data

In [9]:
dataset.columns

Index(['floor_area_sqm', 'floor_area_sqft', 'lease_commence_date', 'house_age',
       'price_per_sqm', 'price_per_sqft', 'resale_price', 'town_BEDOK',
       'town_BISHAN', 'town_BUKIT BATOK', 'town_BUKIT MERAH',
       'town_BUKIT PANJANG', 'town_BUKIT TIMAH', 'town_CENTRAL AREA',
       'town_CHOA CHU KANG', 'town_CLEMENTI', 'town_GEYLANG', 'town_HOUGANG',
       'town_JURONG EAST', 'town_JURONG WEST', 'town_KALLANG/WHAMPOA',
       'town_MARINE PARADE', 'town_PASIR RIS', 'town_PUNGGOL',
       'town_QUEENSTOWN', 'town_SEMBAWANG', 'town_SENGKANG', 'town_SERANGOON',
       'town_TAMPINES', 'town_TOA PAYOH', 'town_WOODLANDS', 'town_YISHUN',
       'flat_type_2 ROOM', 'flat_type_3 ROOM', 'flat_type_4 ROOM',
       'flat_type_5 ROOM', 'flat_type_EXECUTIVE', 'flat_type_MULTI-GENERATION',
       'storey_range_04 TO 06', 'storey_range_07 TO 09',
       'storey_range_10 TO 12', 'storey_range_13 TO 15',
       'storey_range_16 TO 18', 'storey_range_19 TO 21',
       'storey_range_22 TO 24', 

In [10]:
X = dataset.drop("resale_price", axis=1)
y = dataset["resale_price"]

In [11]:
X

,floor_area_sqm,floor_area_sqft,lease_commence_date,house_age,price_per_sqm,price_per_sqft,town_BEDOK,town_BISHAN,town_BUKIT BATOK,town_BUKIT MERAH,...,flat_model_Multi Generation,flat_model_New Generation,flat_model_Premium Apartment,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
0,44.0,473.6116,1979,47,5272.727273,489.852867,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,67.0,721.1813,1978,48,3731.343284,346.653470,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,67.0,721.1813,1980,46,3910.447761,363.292836,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,68.0,731.9452,1980,46,3897.058824,362.048962,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,67.0,721.1813,1980,46,3955.223881,367.452678,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232183,146.0,1571.5294,1988,38,6815.068493,633.141193,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
232184,142.0,1528.4738,1987,39,6901.408451,641.162446,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
232185,145.0,1560.7655,1987,39,6620.689655,615.082791,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
232186,145.0,1560.7655,1988,38,7371.641379,684.848557,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
y

0          232000.0
1          250000.0
2          262000.0
3          265000.0
4          265000.0
            ...    
232183     995000.0
232184     980000.0
232185     960000.0
232186    1068888.0
232187    1120000.0
Name: resale_price, Length: 232188, dtype: float64

In [13]:
training_columns = X.columns.tolist()

# Save the training columns to pass the user input while deployment phase
pickle.dump(training_columns, open("hdb_training_columns.sav", "wb"))

In [14]:
# Split data into training and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
X_train

,floor_area_sqm,floor_area_sqft,lease_commence_date,house_age,price_per_sqm,price_per_sqft,town_BEDOK,town_BISHAN,town_BUKIT BATOK,town_BUKIT MERAH,...,flat_model_Multi Generation,flat_model_New Generation,flat_model_Premium Apartment,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
31681,97.0,1044.0983,2012,14,5536.082474,514.319389,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
187176,130.0,1399.3070,2001,25,5384.615385,500.247623,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
21503,148.0,1593.0572,1986,40,3918.918919,364.079833,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
91473,111.0,1194.7929,2000,26,6846.846847,636.093502,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
34786,131.0,1410.0709,1987,39,3053.435115,283.673679,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119879,67.0,721.1813,1986,40,5000.000000,464.515649,0,0,1,0,...,0,1,0,0,0,0,0,0,0,0
103694,101.0,1087.1539,1998,28,4068.198020,377.948329,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
131932,110.0,1184.0290,2005,21,4681.818182,434.955563,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
146867,92.0,990.2788,2019,7,6467.391304,600.840894,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
X_test

,floor_area_sqm,floor_area_sqft,lease_commence_date,house_age,price_per_sqm,price_per_sqft,town_BEDOK,town_BISHAN,town_BUKIT BATOK,town_BUKIT MERAH,...,flat_model_Multi Generation,flat_model_New Generation,flat_model_Premium Apartment,flat_model_Premium Apartment Loft,flat_model_Premium Maisonette,flat_model_Simplified,flat_model_Standard,flat_model_Terrace,flat_model_Type S1,flat_model_Type S2
185691,93.0,1001.0427,2016,10,6990.193548,649.410859,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
131771,110.0,1184.0290,2003,23,4895.454545,454.803050,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
18106,93.0,1001.0427,2013,13,5000.000000,464.515649,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
12466,65.0,699.6535,1974,52,4123.076923,383.046751,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
53701,116.0,1248.6124,2014,12,5706.896552,530.188552,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33871,92.0,990.2788,2014,12,4565.217391,424.122984,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
92110,143.0,1539.2377,1997,29,3951.048951,367.064814,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
150209,106.0,1140.9734,1988,38,4198.113208,390.017857,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41272,103.0,1108.6817,1988,38,3203.883495,297.650805,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
scaler_x = StandardScaler()
X_train = scaler_x.fit_transform(X_train)
X_test = scaler_x.transform(X_test)

# Use the same scaler in deployment phase, to avoid redundant code
filename = "hdb_scaler_x.sav"
pickle.dump(scaler_x, open(filename, "wb"))

In [18]:
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(
    np.array(y_train).reshape(-1, 1)
).ravel()

y_test = scaler_y.transform(
    np.array(y_test).reshape(-1, 1)
).ravel()

filename = "hdb_scaler_y.sav"
pickle.dump(scaler_y, open(filename, "wb"))

---

### Model building
- DecisionTreeRegressor
- RandomForestRegressor
- SVR (due to take much time, I skip SVR)

In [19]:
# Model 1: DecisionTreeRegressor

In [20]:
grid = {
        "criterion": ['squared_error', 'friedman_mse'],
        "max_features": ["sqrt"],
        "splitter": ["best", "random"]
       }

model_dt = GridSearchCV(DecisionTreeRegressor(), grid, refit = True, verbose = 3, n_jobs = 1)
model_dt.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
[CV 1/5] END criterion=squared_error, max_features=sqrt, splitter=best;, score=0.993 total time=   0.4s
[CV 2/5] END criterion=squared_error, max_features=sqrt, splitter=best;, score=0.991 total time=   0.4s
[CV 3/5] END criterion=squared_error, max_features=sqrt, splitter=best;, score=0.992 total time=   0.3s
[CV 4/5] END criterion=squared_error, max_features=sqrt, splitter=best;, score=0.991 total time=   0.4s
[CV 5/5] END criterion=squared_error, max_features=sqrt, splitter=best;, score=0.990 total time=   0.4s
[CV 1/5] END criterion=squared_error, max_features=sqrt, splitter=random;, score=0.977 total time=   0.4s
[CV 2/5] END criterion=squared_error, max_features=sqrt, splitter=random;, score=0.975 total time=   0.4s
[CV 3/5] END criterion=squared_error, max_features=sqrt, splitter=random;, score=0.979 total time=   0.4s
[CV 4/5] END criterion=squared_error, max_features=sqrt, splitter=random;, score=0.981 total time=   0

,estimator,DecisionTreeRegressor()
,param_grid,"{'criterion': ['squared_error', 'friedman_mse'], 'max_features': ['sqrt'], 'splitter': ['best', 'random']}"
,scoring,None
,n_jobs,1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'squared_error'


In [21]:
# Model 2: RandomForestRegressor

In [22]:
grid = {
        "criterion": ['squared_error', 'friedman_mse'],
        "n_estimators": [10, 25, 50],
        "max_features": ["sqrt", "log2"]
       }

model_rf = GridSearchCV(RandomForestRegressor(), grid, refit = True, verbose = 3, n_jobs = 1)
model_rf.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5] END criterion=squared_error, max_features=sqrt, n_estimators=10;, score=0.997 total time=   2.8s
[CV 2/5] END criterion=squared_error, max_features=sqrt, n_estimators=10;, score=0.998 total time=   2.7s
[CV 3/5] END criterion=squared_error, max_features=sqrt, n_estimators=10;, score=0.997 total time=   2.7s
[CV 4/5] END criterion=squared_error, max_features=sqrt, n_estimators=10;, score=0.998 total time=   2.7s
[CV 5/5] END criterion=squared_error, max_features=sqrt, n_estimators=10;, score=0.998 total time=   2.7s
[CV 1/5] END criterion=squared_error, max_features=sqrt, n_estimators=25;, score=0.998 total time=   6.7s
[CV 2/5] END criterion=squared_error, max_features=sqrt, n_estimators=25;, score=0.998 total time=   6.7s
[CV 3/5] END criterion=squared_error, max_features=sqrt, n_estimators=25;, score=0.998 total time=   6.8s
[CV 4/5] END criterion=squared_error, max_features=sqrt, n_estimators=25;, score=0.998 tota

,estimator,RandomForestRegressor()
,param_grid,"{'criterion': ['squared_error', 'friedman_mse'], 'max_features': ['sqrt', 'log2'], 'n_estimators': [10, 25, ...]}"
,scoring,None
,n_jobs,1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,50


In [23]:
# Model 3: SVR

In [24]:
grid = {
        "kernel": ["linear", "rbf"],
        "C": [10],
        "gamma": ["auto"]
}

# model_svr = GridSearchCV(SVR(), grid, refit = True, verbose = 3, n_jobs = 1)
# model_svr.fit(X_train, y_train)

In [25]:
# Save the model for deployment purpose
pickle.dump(model_dt, open("hdb_model_dt.sav", "wb"))
pickle.dump(model_rf, open("hdb_model_rf.sav", "wb"))
# pickle.dump(model_svr, open("hdb_model_svr.sav", "wb"))

---

### Evaluvation & Model Selection

In [26]:
# R2 Score -> DecisionTreeRegressor
y_pred = model_dt.predict(X_test)
score_dt = r2_score(y_test, y_pred)
print("R2 Score for DecisionTreeRegressor:", score_dt)

R2 Score for DecisionTreeRegressor: 0.9918730779657848


In [27]:
# R2 Score -> RandomForestRegressor
y_pred = model_dt.predict(X_test)
score_rf = r2_score(y_test, y_pred)
print("R2 Score for RandomForestRegressor:", score_rf)

R2 Score for RandomForestRegressor: 0.9918730779657848


In [28]:
# R2 Score -> SVR
# y_pred = model_dt.predict(X_test)
# score_svr = r2_score(y_test, y_pred)
# print("R2 Score for SVR:", score_svr)